# 01 — Baseline Analysis

**Research:** Early-Stage CKD Detection Using Hybrid Optimization (SGPO v2)

**Dataset:** MIMIC-IV CKD Cohort (57,875 patients, 42 features)

**Team:** Muhammed Abdel-Hamid | Abdul Rahman Al-Bili

**Supervisor:** Dr. Ibrahim | New Mansoura University | CSE015

---

This notebook establishes the **baseline Random Forest** performance on the MIMIC-IV CKD dataset, before applying SGPO v2 optimization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

## 1. Load Dataset

In [ ]:
# Update this path for your environment
# Colab: '/content/drive/MyDrive/ckd_project/data/mimic_ckd_dataset_final.csv'
# Local: '../data/processed/mimic_ckd_dataset_final.csv'
DATA_PATH = '../data/processed/mimic_ckd_dataset_final.csv'

df = pd.read_csv(DATA_PATH)
X = df.drop(columns=['subject_id', 'ckd_label'])
y = df['ckd_label']

print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Features: {X.shape[1]}')
print(f'CKD: {y.sum():,} ({y.mean()*100:.1f}%) | Non-CKD: {(y==0).sum():,} ({(1-y.mean())*100:.1f}%)')

## 2. Exploratory Data Analysis

In [ ]:
# Missing values
null_pct = X.isnull().mean().sort_values(ascending=False) * 100
null_pct = null_pct[null_pct > 0]

if len(null_pct) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    null_pct.plot(kind='barh', color='coral', ax=ax)
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Feature')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values.')

In [ ]:
# Key lab distributions by CKD status
key_labs = ['creatinine', 'urea_nitrogen', 'hemoglobin', 'potassium',
            'sodium', 'albumin', 'glucose', 'calcium_total']
key_labs = [lab for lab in key_labs if lab in X.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, lab in enumerate(key_labs):
    ax = axes[i]
    df[df['ckd_label']==0][lab].dropna().hist(bins=40, alpha=0.6, ax=ax, label='Non-CKD', color='steelblue', density=True)
    df[df['ckd_label']==1][lab].dropna().hist(bins=40, alpha=0.6, ax=ax, label='CKD', color='indianred', density=True)
    ax.set_title(lab)
    ax.legend(fontsize=8)
for i in range(len(key_labs), len(axes)):
    axes[i].set_visible(False)
plt.suptitle('Lab Value Distributions: CKD vs Non-CKD', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Baseline Random Forest (5-Fold CV)

In [ ]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Running 5-fold cross-validation ...')
auc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
acc_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
sens_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='recall', n_jobs=-1)

print(f'\nAccuracy:    {acc_scores.mean():.4f} (+/- {acc_scores.std():.4f})')
print(f'AUC-ROC:     {auc_scores.mean():.4f} (+/- {auc_scores.std():.4f})')
print(f'Sensitivity: {sens_scores.mean():.4f} (+/- {sens_scores.std():.4f})')

## 4. Feature Importance

In [ ]:
# Train on full data for feature importance
pipeline.fit(X, y)
importances = pipeline.named_steps['clf'].feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.tail(20).plot(kind='barh', color='teal', ax=ax)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 20 Features — Random Forest Baseline', fontsize=14)
plt.tight_layout()
plt.show()

## 5. ROC Curve & Confusion Matrix

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc_val:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Baseline RF')
axes[0].legend()

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Non-CKD', 'CKD'], cmap='Blues', ax=axes[1])
axes[1].set_title('Confusion Matrix — Baseline RF')

plt.tight_layout()
plt.show()

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Non-CKD', 'CKD']))

## 6. Summary

| Metric | Value |
|---|---|
| Dataset | MIMIC-IV CKD Cohort |
| Patients | 57,875 |
| Features | 42 |
| Model | Random Forest (100 trees) |
| AUC-ROC | 0.9561 +/- 0.0011 |
| Sensitivity | 0.8904 +/- 0.0034 |
| Accuracy | 0.8897 +/- 0.0022 |

**Next:** Apply SGPO v2 optimization to reduce features and tune hyperparameters.